In [1]:
import joblib

X_train, X_test, y_train, y_test = joblib.load(
    "ds52-train_test_split.joblib"
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (36168, 15)
X_test: (9043, 15)
y_train: (36168,)
y_test: (9043,)


## Feature Engineering

New features are created using information available before making the marketing decision. The same transformations are applied independently to the training and test datasets. The target variable is not used during feature creation, preventing target leakage.

1.1 import libraries

In [2]:
import pandas as pd
import numpy as np


1.2 protects member 1 original dataset.

In [3]:
X_train_fe = X_train.copy()
X_test_fe = X_test.copy()

1.3 The `pdays` value of `-1` means that the customer was not contacted in a previous campaign. A binary feature is created to represent this information clearly.

In [4]:
X_train_fe["previously_contacted"] = (
    X_train_fe["pdays"] != -1
).astype(int)

In [5]:
X_test_fe["previously_contacted"] = (
    X_test_fe["pdays"] != -1
).astype(int)

In [6]:
print(X_train_fe["previously_contacted"].value_counts())

previously_contacted
0    29584
1     6584
Name: count, dtype: int64


In [7]:
X_train_fe[["pdays", "previously_contacted"]].head()

,pdays,previously_contacted
24001,-1,0
43409,185,1
20669,-1,0
18810,-1,0
23130,-1,0


In [8]:
X_train_fe.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,previously_contacted
24001,36,technician,divorced,secondary,no,861,no,no,telephone,29,aug,2,-1,0,unknown,0
43409,24,student,single,secondary,no,4126,no,no,cellular,5,apr,4,185,7,failure,1
20669,44,technician,single,secondary,no,244,yes,no,cellular,12,aug,4,-1,0,unknown,0
18810,48,unemployed,married,secondary,no,0,no,no,telephone,31,jul,11,-1,0,unknown,0
23130,38,technician,married,secondary,no,257,no,no,cellular,26,aug,10,-1,0,unknown,0


1.4 Clean Previous Contact Days

The value `-1` indicates that the customer was never previously contacted. It is replaced with `0`, while `previously_contacted` preserves whether an earlier contact occurred.

In [9]:
X_train_fe["pdays_clean"] = X_train_fe["pdays"].replace(-1, 0)

In [10]:
X_test_fe["pdays_clean"] = X_test_fe["pdays"].replace(-1, 0)

In [11]:
X_train_fe[
    ["pdays", "pdays_clean", "previously_contacted"]
].head(10)

,pdays,pdays_clean,previously_contacted
24001,-1,0,0
43409,185,185,1
20669,-1,0,0
18810,-1,0,0
23130,-1,0,0
15058,-1,0,0
15908,-1,0,0
30424,-1,0,0
9998,-1,0,0
14935,-1,0,0


1.5 Customer Loan Count and Credit Default

The housing and personal loan indicators are combined into `loan_count`. Credit default is represented separately because it describes repayment status rather than an additional loan.

In [12]:
X_train_fe["loan_count"] = (
    (X_train_fe["housing"] == "yes").astype(int)
    + (X_train_fe["loan"] == "yes").astype(int)
)

In [13]:
X_test_fe["loan_count"] = (
    (X_test_fe["housing"] == "yes").astype(int)
    + (X_test_fe["loan"] == "yes").astype(int)
)

In [14]:
X_train_fe["has_credit_default"] = (
    X_train_fe["default"] == "yes"
).astype(int)

In [15]:
X_test_fe["has_credit_default"] = (
    X_test_fe["default"] == "yes"
).astype(int)

In [16]:
print("Loan count:")
print(X_train_fe["loan_count"].value_counts().sort_index())

print("\nCredit default:")
print(X_train_fe["has_credit_default"].value_counts().sort_index())

Loan count:
loan_count
0    13702
1    18937
2     3529
Name: count, dtype: int64

Credit default:
has_credit_default
0    35521
1      647
Name: count, dtype: int64


1.6 Previous Campaign Success


Creating a new feature.This binary feature indicates whether the customer responded successfully to a previous marketing campaign.

In [17]:
X_train_fe["previous_campaign_success"] = (
    X_train_fe["poutcome"] == "success"
).astype(int)

In [18]:
X_test_fe["previous_campaign_success"] = (
    X_test_fe["poutcome"] == "success"
).astype(int)

In [19]:
print(X_train_fe["previous_campaign_success"].value_counts())

previous_campaign_success
0    34963
1     1205
Name: count, dtype: int64


1.7 Customer Age Group

Customers are placed into age groups to capture possible non-linear differences in campaign response between younger, middle-aged, and older customers.

In [20]:
age_bins = [17, 30, 40, 50, 60, np.inf]

In [21]:


age_labels = [
    "18-30",
    "31-40",
    "41-50",
    "51-60",
    "60+"
]

In [22]:
X_train_fe["age_group"] = pd.cut(
    X_train_fe["age"],
    bins=age_bins,
    labels=age_labels
)

In [23]:
X_test_fe["age_group"] = pd.cut(
    X_test_fe["age"],
    bins=age_bins,
    labels=age_labels
)

In [24]:

print(X_train_fe["age_group"].value_counts().sort_index())

age_group
18-30     5703
31-40    14135
41-50     8983
51-60     6378
60+        969
Name: count, dtype: int64


In [25]:
X_train_fe[["age", "age_group"]].head(10)

,age,age_group
24001,36,31-40
43409,24,18-30
20669,44,41-50
18810,48,41-50
23130,38,31-40
15058,48,41-50
15908,50,41-50
30424,46,41-50
9998,46,41-50
14935,45,41-50


1.8 Customer Balance Group

Account balances are grouped into meaningful financial categories while the original numerical balance is retained.

In [26]:
balance_bins = [-np.inf, -1, 0, 1000, 5000, np.inf]

In [27]:
balance_labels = [
    "negative",
    "zero",
    "low",
    "medium",
    "high"
]

In [28]:
X_train_fe["balance_group"] = pd.cut(
    X_train_fe["balance"],
    bins=balance_bins,
    labels=balance_labels
)

In [29]:
X_test_fe["balance_group"] = pd.cut(
    X_test_fe["balance"],
    bins=balance_bins,
    labels=balance_labels
)

In [30]:
X_train_fe[["balance", "balance_group"]].head(10)

,balance,balance_group
24001,861,low
43409,4126,medium
20669,244,low
18810,0,zero
23130,257,low
15058,1513,medium
15908,4315,medium
30424,-780,negative
9998,474,low
14935,248,low


In [31]:
X_train_fe["balance_group"].value_counts().sort_index()

balance_group
negative     3001
zero         2809
low         18614
medium       9463
high         2281
Name: count, dtype: int64

1.9 Campaign Contact Group

The number of contacts made during the campaign is grouped to represent the level of contact effort.

In [32]:
campaign_bins = [0, 1, 2, 4, np.inf]

In [33]:
campaign_labels = [
    "first_contact",
    "second_contact",
    "three_to_four",
    "five_or_more"
]

In [34]:
X_train_fe["campaign_contact_group"] = pd.cut(
    X_train_fe["campaign"],
    bins=campaign_bins,
    labels=campaign_labels
)

In [35]:
X_test_fe["campaign_contact_group"] = pd.cut(
    X_test_fe["campaign"],
    bins=campaign_bins,
    labels=campaign_labels
)

In [36]:
X_train_fe[
    ["campaign", "campaign_contact_group"]
].head(10)

,campaign,campaign_contact_group
24001,2,second_contact
43409,4,three_to_four
20669,4,three_to_four
18810,11,five_or_more
23130,10,five_or_more
15058,1,first_contact
15908,11,five_or_more
30424,1,first_contact
9998,2,second_contact
14935,2,second_contact


In [37]:
X_train_fe[
    "campaign_contact_group"
].value_counts().sort_index()

campaign_contact_group
first_contact     14026
second_contact    10023
three_to_four      7217
five_or_more       4902
Name: count, dtype: int64

1.10 Feature Engineering Verification

In [38]:
new_features = [
    column for column in X_train_fe.columns
    if column not in X_train.columns
]

In [39]:
print("Original training shape :", X_train.shape)
print("Engineered training shape:", X_train_fe.shape)

Original training shape : (36168, 15)
Engineered training shape: (36168, 23)


In [40]:
print("Original testing shape  :", X_test.shape)
print("Engineered testing shape :", X_test_fe.shape)

Original testing shape  : (9043, 15)
Engineered testing shape : (9043, 23)


In [41]:
print("\nNew features:")

for feature in new_features:
    print("-", feature)


New features:
- previously_contacted
- pdays_clean
- loan_count
- has_credit_default
- previous_campaign_success
- age_group
- balance_group
- campaign_contact_group


In [42]:
print("\nMissing values in training data:")
print(X_train_fe.isnull().sum().sum())


Missing values in training data:
0


In [43]:
print("\nMissing values in testing data:")
print(X_test_fe.isnull().sum().sum())


Missing values in testing data:
0


In [44]:
same_columns = X_train_fe.columns.equals(X_test_fe.columns)

print("Training and testing columns match:", same_columns)

Training and testing columns match: True


In [45]:
X_train_fe.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,...,previous,poutcome,previously_contacted,pdays_clean,loan_count,has_credit_default,previous_campaign_success,age_group,balance_group,campaign_contact_group
24001,36,technician,divorced,secondary,no,861,no,no,telephone,29,...,0,unknown,0,0,0,0,0,31-40,low,second_contact
43409,24,student,single,secondary,no,4126,no,no,cellular,5,...,7,failure,1,185,0,0,0,18-30,medium,three_to_four
20669,44,technician,single,secondary,no,244,yes,no,cellular,12,...,0,unknown,0,0,1,0,0,41-50,low,three_to_four
18810,48,unemployed,married,secondary,no,0,no,no,telephone,31,...,0,unknown,0,0,0,0,0,41-50,zero,five_or_more
23130,38,technician,married,secondary,no,257,no,no,cellular,26,...,0,unknown,0,0,0,0,0,31-40,low,five_or_more


## 2. Verify Engineered Features

In [46]:
expected_features = [
    "previously_contacted",
    "pdays_clean",
    "loan_count",
    "has_credit_default",
    "previous_campaign_success",
    "age_group",
    "balance_group",
    "campaign_contact_group"
]

In [47]:
missing_features = [
    feature for feature in expected_features
    if feature not in X_train_fe.columns
]

unexpected_features = [
    feature for feature in X_train_fe.columns
    if feature not in X_train.columns
    and feature not in expected_features
]

print("Missing features:", missing_features)
print("Unexpected features:", unexpected_features)

Missing features: []
Unexpected features: []


In [48]:
print("Original training shape:", X_train.shape)
print("Engineered training shape:", X_train_fe.shape)

print("Original testing shape:", X_test.shape)
print("Engineered testing shape:", X_test_fe.shape)

Original training shape: (36168, 15)
Engineered training shape: (36168, 23)
Original testing shape: (9043, 15)
Engineered testing shape: (9043, 23)


In [49]:
print(
    "Training and testing columns match:",
    X_train_fe.columns.equals(X_test_fe.columns)
)

print(
    "Missing training values:",
    X_train_fe.isnull().sum().sum()
)

print(
    "Missing testing values:",
    X_test_fe.isnull().sum().sum()
)

Training and testing columns match: True
Missing training values: 0
Missing testing values: 0


In [50]:
X_train_fe[expected_features].head()

,previously_contacted,pdays_clean,loan_count,has_credit_default,previous_campaign_success,age_group,balance_group,campaign_contact_group
24001,0,0,0,0,0,31-40,low,second_contact
43409,1,185,0,0,0,18-30,medium,three_to_four
20669,0,0,1,0,0,41-50,low,three_to_four
18810,0,0,0,0,0,41-50,zero,five_or_more
23130,0,0,0,0,0,31-40,low,five_or_more


## 3. ColumnTransformer Assembly

3.1 Define numerical and categorical columns

In [51]:
numeric_features = (
    X_train_fe
    .select_dtypes(include=["number"])
    .columns
    .drop("pdays")
    .tolist()
)


In [52]:
categorical_features = (
    X_train_fe
    .select_dtypes(exclude=["number"])
    .columns
    .tolist()
)

In [53]:
print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['age', 'balance', 'day', 'campaign', 'previous', 'previously_contacted', 'pdays_clean', 'loan_count', 'has_credit_default', 'previous_campaign_success']

Categorical features:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'age_group', 'balance_group', 'campaign_contact_group']


3.2 Create numerical preprocessing pipeline

In [54]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

3.3 Create categorical preprocessing pipeline

In [55]:
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

3.4 Assemble the ColumnTransfer

In [56]:
from sklearn.compose import ColumnTransformer

feature_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

feature_preprocessor

,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


3.5 Add reference category

In [57]:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            drop="first",
            sparse_output=False
        ))
    ]
)

3.6 Fit only on training data

In [58]:
X_train_transformed = feature_preprocessor.fit_transform(X_train_fe)
X_test_transformed = feature_preprocessor.transform(X_test_fe)

3.7 Verify transformed data

In [59]:
print("Transformed training shape:", X_train_transformed.shape)
print("Transformed testing shape:", X_test_transformed.shape)

print(
    "Same number of columns:",
    X_train_transformed.shape[1] == X_test_transformed.shape[1]
)

print(
    "Training contains NaN:",
    np.isnan(X_train_transformed).any()
)

print(
    "Testing contains NaN:",
    np.isnan(X_test_transformed).any()
)

Transformed training shape: (36168, 68)
Transformed testing shape: (9043, 68)
Same number of columns: True
Training contains NaN: False
Testing contains NaN: False
